# WanderBricks: Property Sentiment Alerts
Daily job that identifies properties with critically high negative sentiment and flags them for property management review.

In [0]:
# UPDATE THE TABLE REFERENCES TO YOUR OWN SCHEMAS

In [0]:
# Table references from the process_customer_reviews pipeline
sentiment_summary_table = "dais2026_2.dev_tanishq_maheshwari_schema.property_sentiment_summary"
alerts_table = "dais2026_2.dev_tanishq_maheshwari_schema.property_sentiment_alerts"
properties_table = "samples.wanderbricks.properties"
hosts_table = "samples.wanderbricks.hosts"

print(f"Sentiment summary : {sentiment_summary_table}")
print(f"Alerts output     : {alerts_table}")
print(f"Properties source : {properties_table}")
print(f"Hosts source      : {hosts_table}")

In [0]:
at_risk_query = """
WITH aggregated AS (
  SELECT
    property_id,
    title,
    destination,
    country,
    COUNT(*) AS total_reviews,
    SUM(CASE WHEN sentiment_label = 'Negative' THEN 1 ELSE 0 END) AS negative_count,
    SUM(CASE WHEN sentiment_label = 'Positive' THEN 1 ELSE 0 END) AS positive_count,
    ROUND(SUM(CASE WHEN sentiment_label = 'Negative' THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS negative_pct,
    ROUND(AVG(composite_score), 2) AS avg_composite_score,
    ROUND(AVG(rating), 2) AS avg_rating,
    MAX(review_date) AS latest_review
  FROM {sentiment_summary_table}
  GROUP BY property_id, title, destination, country
)
SELECT * FROM aggregated
WHERE total_reviews >= 3 AND negative_pct >= 40
ORDER BY negative_pct DESC
"""

at_risk_df = spark.sql(at_risk_query.format(sentiment_summary_table=sentiment_summary_table))
at_risk_df.display()

In [0]:
from datetime import date
from pyspark.sql import functions as F

run_date = date.today()

# Look up host contact info via properties → hosts
properties_df = spark.table(properties_table).select("property_id", "host_id")
hosts_df = spark.table(hosts_table).select(
    "host_id",
    F.col("name").alias("host_name"),
    F.col("email").alias("host_email"),
    F.col("phone").alias("host_phone"),
    "is_verified",
)

# Enrich with severity, run date, and host contact info
enriched_df = (
    at_risk_df
    .withColumn(
        "severity",
        F.when(F.col("negative_pct") >= 60, "CRITICAL").otherwise("WARNING")
    )
    .withColumn("run_date", F.lit(str(run_date)))
    .join(properties_df, on="property_id", how="left")
    .join(hosts_df, on="host_id", how="left")
)

# Append to Delta table — preserves daily history
(
    enriched_df
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(alerts_table)
)

rows = enriched_df.collect()
print(f"✓ {len(rows)} at-risk properties written to {alerts_table} (run_date: {run_date})")
print(f"  Host email resolved for {sum(1 for r in rows if r['host_email'])} / {len(rows)} properties")

In [0]:
def build_email(row):
    """Format a property owner alert email."""
    severity = row["severity"]
    subject = f"[WanderBricks] {severity}: Action Required for '{row['title']}'"
    action = (
        "Immediate attention is required \u2014 guest satisfaction is critically low."
        if severity == "CRITICAL"
        else "We recommend reviewing recent guest feedback and considering improvements."
    )
    host_name = row["host_name"] or "Property Partner"
    body = f"""
Dear {host_name},

We are writing to inform you of a guest sentiment alert for your property listing.

{'─' * 38}
Property   : {row['title']}
Location   : {row['destination']}, {row['country']}
Alert Level: {severity}
{'─' * 38}
Negative Review Rate : {row['negative_pct']}%
Average Guest Rating : {row['avg_rating']} / 5.0
Total Reviews        : {row['total_reviews']}
Latest Review        : {str(row['latest_review'])[:10]}
{'─' * 38}

{action}

Please log in to your WanderBricks Host Dashboard to view detailed guest
feedback and take the recommended steps to improve your property's performance.

Best regards,
WanderBricks Quality & Guest Experience Team
support@wanderbricks.com
"""
    return subject, body


def simulate_send(to_email, host_name, subject, body):
    """Simulate sending an email (replace with SMTP / SendGrid in production)."""
    print(f"{'─' * 60}")
    print(f"  TO      : {to_email} ({host_name})")
    print(f"  SUBJECT : {subject}")
    print(f"  BODY    :")
    for line in body.split("\n"):
        print(f"           {line}")


print("=== Simulated Email Dispatch ===\n")

for row in rows:
    host_email = row["host_email"] or f"owner_{row['property_id']}@wanderbricks.com"
    host_name  = row["host_name"]  or "Property Partner"
    subject, body = build_email(row)
    simulate_send(host_email, host_name, subject, body)

print(f"\n{'─' * 60}")
print(f"\u2713 Dispatch complete \u2014 {len(rows)} alert emails simulated.")
print(f"{'─' * 60}")

dbutils.notebook.exit(f"{len(rows)} properties flagged and notified")